In [ ]:
try:
    import pyspark.sql.functions as F
    from pyspark.sql import Window
    from pyspark.sql.types import IntegerType
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

SCHEMA_ORIGEM = "bronze"
TABELA_ORIGEM = "tb_movies_info"
SCHEMA_DESTINO = "silver"
TABELA_DESTINO = "tb_info_filmes"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

# garante que a coluna tecnica publicada na Silver esteja em portugues e com o tipo correto
def validar_coluna_ingestao_silver(df, nome_tabela):
    if "ingestion_datetime" in df.columns:
        raise AssertionError(f"A tabela {nome_tabela} ainda possui a coluna ingestion_datetime.")
    tipo_ingestao = dict(df.dtypes).get("data_hora_ingestao")
    if tipo_ingestao != "timestamp":
        raise AssertionError(
            f"A tabela {nome_tabela} deve possuir data_hora_ingestao como TIMESTAMP; tipo encontrado: {tipo_ingestao}."
        )

In [ ]:
df_bronze = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_ORIGEM}")

colunas_esperadas = {
    "id", "tconst", "title", "original_title",
    "original_language", "release_date", "runtime",
    "status", "overview", "tagline", "ingestion_datetime"
}
colunas_ausentes = colunas_esperadas.difference(df_bronze.columns)
if colunas_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_ausentes)}")

# mantém a bronze inalterada e aplica as transformações em uma cópia
df_tratado = (
    df_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("id_imdb", F.trim(F.col("tconst")))
    .withColumn("titulo", F.trim(F.col("title")))
    .withColumn("titulo_original", F.trim(F.col("original_title")))
    .withColumn("idioma_original", F.trim(F.col("original_language")))
    .withColumn("duracao_minutos", F.expr("try_cast(runtime AS INT)"))
    .withColumn("sinopse", F.trim(F.col("overview")))
    .withColumn("frase_divulgacao", F.trim(F.col("tagline")))
)

In [ ]:
# normaliza caixa, espaços e hífens antes de traduzir os status
status_normalizado = F.lower(
    F.trim(F.regexp_replace(F.coalesce(F.col("status"), F.lit("")), r"[\s_-]+", " "))
)
df_tratado = df_tratado.withColumn("status_normalizado", status_normalizado)
df_tratado = df_tratado.withColumn(
    "status_filme",
    F.when(F.col("status_normalizado") == "released", F.lit("Lançado"))
     .when(F.col("status_normalizado") == "post production", F.lit("Pós-Produção"))
     .when(F.col("status_normalizado") == "in production", F.lit("Em Produção"))
     .when(F.col("status_normalizado") == "planned", F.lit("Planejado"))
     .when(F.col("status_normalizado") == "rumored", F.lit("Rumores"))
     .when(F.col("status_normalizado") == "canceled", F.lit("Cancelado"))
     .otherwise(F.lit("Não Informado"))
).drop("status_normalizado")

In [ ]:
# converte formatos conhecidos com funcoes tolerantes; valores impossiveis permanecem NULL
def converter_data_multiformato(nome_coluna):
    return F.coalesce(
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'yyyy-MM-dd')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'dd/MM/yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'dd-MM-yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'MM-dd-yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'yyyy/MM/dd')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'dd.MM.yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'yyyyMMdd')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna})")),
    )

df_tratado = (
    df_tratado
    .withColumn("data_texto", F.trim(F.col("release_date")))
    .withColumn("data_lancamento", converter_data_multiformato("data_texto"))
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")).cast(IntegerType()))
)
df_datas_nao_convertidas = (
    df_tratado
    .where(F.col("data_texto").isNotNull() & (F.col("data_texto") != "") & F.col("data_lancamento").isNull())
    .select("id_filme", F.col("release_date").alias("data_origem"), "data_texto")
    .dropDuplicates(["id_filme", "data_texto"])
)

# mantém a versão mais recente de cada filme conforme a ingestão
janela_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("data_hora_ingestao").desc())
df_tratado = (
    df_tratado
    .withColumn("ordem_ingestao", F.row_number().over(janela_mais_recente))
    .where(F.col("id_filme").isNotNull() & (F.col("ordem_ingestao") == 1))
    .drop("ordem_ingestao", "id", "tconst", "title", "original_title",
          "original_language", "release_date", "data_texto", "runtime", "overview",
          "ingestion_datetime")
)

colunas_silver = [
    "id_filme", "id_imdb", "titulo", "titulo_original",
    "idioma_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "status_filme", "sinopse", "frase_divulgacao",
    "data_hora_ingestao"
]
df_silver = df_tratado.select(*colunas_silver)

In [ ]:
# grava a silver em overwrite para permitir reprocessamento idempotente
(df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}").limit(10))

In [ ]:
# valida unicidade, formatos de data, coerencia do ano e valores nao interpretaveis
df_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}")
validar_coluna_ingestao_silver(df_validacao, TABELA_DESTINO)
casos_teste_datas = [
    ("2024-03-15", "2024-03-15"),
    ("15/03/2024", "2024-03-15"),
    ("15-03-2024", "2024-03-15"),
    ("03-15-2024", "2024-03-15"),
    ("2024/03/15", "2024-03-15"),
    ("15.03.2024", "2024-03-15"),
    ("20240315", "2024-03-15"),
    ("2024-03-15 10:30:00", "2024-03-15"),
    ("data inválida", None),
]
df_teste_datas = (
    spark.createDataFrame(casos_teste_datas, ["data_texto", "data_esperada"])
    .withColumn("data_convertida", converter_data_multiformato("data_texto"))
    .withColumn("data_resultado", F.date_format("data_convertida", "yyyy-MM-dd"))
)
formatos_data_invalidos = df_teste_datas.where(
    ~F.col("data_resultado").eqNullSafe(F.col("data_esperada"))
).count()
duplicados = (
    df_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
datas_nulas = (
    df_validacao.where(F.col("data_lancamento").isNull()).count()
)
anos_inconsistentes = df_validacao.where(
    F.col("data_lancamento").isNotNull()
    & (F.col("ano_lancamento") != F.year(F.col("data_lancamento")))
).count()

if formatos_data_invalidos != 0:
    raise AssertionError(f"Há {formatos_data_invalidos} formatos de data convertidos incorretamente nos testes.")
if duplicados != 0:
    raise AssertionError(f"Há {duplicados} ids de filme duplicados na Silver.")
if anos_inconsistentes != 0:
    raise AssertionError(f"Há {anos_inconsistentes} anos de lançamento inconsistentes com a data.")

display(df_datas_nao_convertidas.orderBy("data_texto").limit(20))
display(
    df_validacao.groupBy("status_filme").count().orderBy(F.col("count").desc())
)
print(f"Registros Silver: {df_validacao.count()}")
print(f"Datas nulas (inválidas ou ausentes): {datas_nulas}")
print(f"Datas de origem não convertidas: {df_datas_nao_convertidas.count()}")
print(f"Formatos inválidos nos testes de data: {formatos_data_invalidos}")
print(f"Anos inconsistentes com a data: {anos_inconsistentes}")
print(f"Duplicidades por id_filme: {duplicados}")

In [ ]:
SCHEMA_FINANCEIRO_ORIGEM = "bronze"
TABELA_FINANCEIRO_ORIGEM = "tb_movies_financials"
TABELA_COTACAO_ORIGEM = "tb_cotacao_dolar"
TABELA_FINANCEIRO_DESTINO = "tb_financeiro_filmes"

df_financeiro_bronze = spark.table(f"{SCHEMA_FINANCEIRO_ORIGEM}.{TABELA_FINANCEIRO_ORIGEM}")
colunas_financeiras_esperadas = {"id", "budget", "revenue", "ingestion_datetime"}
colunas_financeiras_ausentes = colunas_financeiras_esperadas.difference(df_financeiro_bronze.columns)
if colunas_financeiras_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_financeiras_ausentes)}")

In [ ]:
# normaliza valores monetarios sem UDF, preservando o processamento distribuido no Spark
marcadores_ausencia = r"^(|unknown|não informado|na|n/a|null)$"

def normalizar_valor_monetario(coluna):
    texto = F.lower(F.trim(coluna.cast("string")))
    valor_sem_simbolos = F.regexp_replace(texto, r"[^0-9,.\-]", "")
    valor_sem_simbolos = F.when(
        texto.rlike(r"^\s*\(.*\)\s*$"),
        F.concat(F.lit("-"), valor_sem_simbolos),
    ).otherwise(valor_sem_simbolos)

    possui_virgula = F.instr(valor_sem_simbolos, ",") > 0
    possui_ponto = F.instr(valor_sem_simbolos, ".") > 0
    ultimo_separador = F.regexp_extract(valor_sem_simbolos, r"([.,])[0-9]+$", 1)
    virgula_como_milhar = valor_sem_simbolos.rlike(r"^-?[0-9]{1,3}(,[0-9]{3})+$")
    ponto_como_milhar = valor_sem_simbolos.rlike(r"^-?[0-9]{1,3}(\.[0-9]{3})+$")

    valor_normalizado = (
        F.when(possui_virgula & possui_ponto & (ultimo_separador == ","),
               F.regexp_replace(F.regexp_replace(valor_sem_simbolos, r"\.", ""), ",", "."))
         .when(possui_virgula & possui_ponto & (ultimo_separador == "."),
               F.regexp_replace(valor_sem_simbolos, ",", ""))
         .when(possui_virgula & ~possui_ponto & virgula_como_milhar,
               F.regexp_replace(valor_sem_simbolos, ",", ""))
         .when(possui_virgula & ~possui_ponto,
               F.regexp_replace(valor_sem_simbolos, ",", "."))
         .when(possui_ponto & ~possui_virgula & ponto_como_milhar,
               F.regexp_replace(valor_sem_simbolos, r"\.", ""))
         .otherwise(valor_sem_simbolos)
    )

    return F.when(
        texto.isNull() | texto.rlike(marcadores_ausencia) | (valor_sem_simbolos == ""),
        F.lit(None).cast("string"),
    ).otherwise(valor_normalizado)

# renomeia e higieniza os campos da Bronze sem alterar a tabela de origem
df_financeiro = (
    df_financeiro_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("orcamento_normalizado", normalizar_valor_monetario(F.col("budget")))
    .withColumn("receita_normalizada", normalizar_valor_monetario(F.col("revenue")))
    .withColumn("orcamento_usd", F.expr("try_cast(orcamento_normalizado AS DECIMAL(18,2))"))
    .withColumn("receita_usd", F.expr("try_cast(receita_normalizada AS DECIMAL(18,2))"))
)

# valores nulos, zerados ou negativos não representam um valor financeiro válido
df_financeiro = (
    df_financeiro
    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") > 0, F.col("receita_usd")))
)

In [ ]:
df_cotacao = (
    spark.table(f"{SCHEMA_FINANCEIRO_ORIGEM}.{TABELA_COTACAO_ORIGEM}")
    .withColumn("data_hora_cotacao", F.to_timestamp("dataHoraCotacao"))
)

# como a origem financeira não possui data da transação, usa-se a cotação PTAX mais recente disponível
df_cotacao_atual = (
    df_cotacao
    .where(F.col("cotacaoCompra") > 0)
    .orderBy(F.col("data_hora_cotacao").desc())
    .limit(1)
    .select(F.col("cotacaoCompra").cast("DECIMAL(12,6)").alias("cotacao_dolar_brl"))
)

# o cruzamento aplica a mesma taxa de referência a cada filme e mantém o cálculo no cluster
df_financeiro = (
    df_financeiro
    .crossJoin(df_cotacao_atual)
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(18,2)"))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("DECIMAL(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("DECIMAL(18,2)"))
    .withColumn("margem_lucro_percentual", F.when(F.col("receita_usd") > 0, (F.col("lucro_usd") / F.col("receita_usd") * 100).cast("DECIMAL(10,2)")))
)

In [ ]:
colunas_financeiro_silver = [
    "id_filme", "orcamento_usd", "receita_usd",
    "cotacao_dolar_brl", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual",
    "data_hora_ingestao"
]

# mantém somente a versão mais recente de cada filme após as cargas append da bronze
janela_financeira_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("data_hora_ingestao").desc())
df_financeiro_silver = (
    df_financeiro
    .where(F.col("id_filme").isNotNull())
    .withColumn("ordem_ingestao", F.row_number().over(janela_financeira_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .drop("ordem_ingestao")
    .select(*colunas_financeiro_silver)
)

# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_financeiro_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}").limit(10))

In [ ]:
# valida identificadores, formatos monetarios, conversoes e ausencia de divisao por receita nula
df_financeiro_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}")
validar_coluna_ingestao_silver(df_financeiro_validacao, TABELA_FINANCEIRO_DESTINO)
casos_teste_monetarios = [
    ("$1,234.56", "1234.56"),
    ("US$ 1,234.56", "1234.56"),
    ("R$ 1.234,56", "1234.56"),
    ("1.234,56", "1234.56"),
    ("1,234", "1234"),
    ("1234,56", "1234.56"),
    ("1.234", "1234"),
    ("1234.56", "1234.56"),
    ("(1,234.56)", "-1234.56"),
    ("Unknown", None),
    ("Não Informado", None),
    ("N/A", None),
]
df_teste_monetario = (
    spark.createDataFrame(casos_teste_monetarios, ["valor_origem", "valor_esperado"])
    .withColumn("valor_normalizado", normalizar_valor_monetario(F.col("valor_origem")))
)
formatos_monetarios_invalidos = df_teste_monetario.where(
    ~F.col("valor_normalizado").eqNullSafe(F.col("valor_esperado"))
).count()
duplicados_financeiros = (
    df_financeiro_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
valores_financeiros_nao_positivos = df_financeiro_validacao.where(
    (F.col("orcamento_usd").isNotNull() & (F.col("orcamento_usd") <= 0))
    | (F.col("receita_usd").isNotNull() & (F.col("receita_usd") <= 0))
).count()
if formatos_monetarios_invalidos != 0:
    raise AssertionError(f"Há {formatos_monetarios_invalidos} formatos monetários normalizados incorretamente.")
if duplicados_financeiros != 0:
    raise AssertionError(f"Há {duplicados_financeiros} ids de filme duplicados na Silver financeira.")
if valores_financeiros_nao_positivos != 0:
    raise AssertionError(f"Há {valores_financeiros_nao_positivos} valores financeiros não positivos na Silver.")

display(df_financeiro_validacao.select(
    "id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "margem_lucro_percentual"
).limit(10))
print(f"Registros Silver financeira: {df_financeiro_validacao.count()}")
print(f"Orçamentos válidos: {df_financeiro_validacao.where(F.col("orcamento_usd").isNotNull()).count()}")
print(f"Receitas válidas: {df_financeiro_validacao.where(F.col("receita_usd").isNotNull()).count()}")
print(f"Registros sem receita válida: {df_financeiro_validacao.where(F.col("receita_usd").isNull()).count()}")
print(f"Registros sem orçamento válido: {df_financeiro_validacao.where(F.col("orcamento_usd").isNull()).count()}")
print(f"Duplicidades por id_filme: {duplicados_financeiros}")
print(f"Formatos monetários inválidos nos testes: {formatos_monetarios_invalidos}")
print(f"Valores financeiros não positivos: {valores_financeiros_nao_positivos}")

In [ ]:
SCHEMA_METRICAS_ORIGEM = "bronze"
TABELA_METRICAS_ORIGEM = "tb_movies_metrics"
TABELA_METRICAS_DESTINO = "tb_metricas_engajamento"

df_metricas_bronze = spark.table(f"{SCHEMA_METRICAS_ORIGEM}.{TABELA_METRICAS_ORIGEM}")
colunas_metricas_esperadas = {"id", "popularity", "vote_average", "vote_count", "averageRating", "numVotes", "ingestion_datetime"}
colunas_metricas_ausentes = colunas_metricas_esperadas.difference(df_metricas_bronze.columns)
if colunas_metricas_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_metricas_ausentes)}")

In [ ]:
# normaliza decimais aceitando ponto ou virgula sem aceitar textos fora de contexto
def normalizar_numero_decimal(coluna):
    texto = F.lower(F.trim(coluna.cast("string")))
    valor = F.regexp_replace(texto, r"\s+", "")
    formato_numerico = valor.rlike(r"^-?[0-9]+([.,][0-9]+)*$")
    quantidade_virgulas = F.length(valor) - F.length(F.regexp_replace(valor, ",", ""))
    quantidade_pontos = F.length(valor) - F.length(F.regexp_replace(valor, r"\.", ""))
    possui_virgula = quantidade_virgulas > 0
    possui_ponto = quantidade_pontos > 0
    ultimo_separador = F.regexp_extract(valor, r"([.,])[0-9]+$", 1)

    valor_normalizado = (
        F.when(possui_virgula & possui_ponto & (ultimo_separador == ","),
               F.regexp_replace(F.regexp_replace(valor, r"\.", ""), ",", "."))
         .when(possui_virgula & possui_ponto & (ultimo_separador == "."),
               F.regexp_replace(valor, ",", ""))
         .when(possui_virgula & ~possui_ponto & (quantidade_virgulas == 1),
               F.regexp_replace(valor, ",", "."))
         .when(possui_virgula & ~possui_ponto, F.regexp_replace(valor, ",", ""))
         .when(possui_ponto & ~possui_virgula & (quantidade_pontos > 1),
               F.regexp_replace(valor, r"\.", ""))
         .otherwise(valor)
    )
    return F.when(texto.isNull() | ~formato_numerico, F.lit(None).cast("string")).otherwise(valor_normalizado)

# contagens aceitam inteiros simples, agrupamentos de milhar e representacoes terminadas em .0 ou ,0
def normalizar_numero_inteiro(coluna):
    texto = F.lower(F.trim(coluna.cast("string")))
    valor = F.regexp_replace(texto, r"\s+", "")
    inteiro_simples = valor.rlike(r"^-?[0-9]+$")
    inteiro_com_milhar = valor.rlike(r"^-?[0-9]{1,3}([.,][0-9]{3})+$")
    inteiro_com_decimal_zero = valor.rlike(r"^-?[0-9]+[.,]0+$")
    return (
        F.when(texto.isNull(), F.lit(None).cast("string"))
         .when(inteiro_simples, valor)
         .when(inteiro_com_milhar, F.regexp_replace(valor, r"[.,]", ""))
         .when(inteiro_com_decimal_zero, F.regexp_replace(valor, r"[.,]0+$", ""))
         .otherwise(F.lit(None).cast("string"))
    )

# renomeia os campos para o padrao da Silver sem alterar a tabela Bronze
df_metricas = (
    df_metricas_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("popularidade_texto", F.trim(F.col("popularity")))
    .withColumn("nota_media_tmdb_texto", F.trim(F.col("vote_average")))
    .withColumn("qtd_votos_tmdb_texto", F.trim(F.col("vote_count")))
    .withColumn("nota_media_imdb_texto", F.trim(F.col("averageRating")))
    .withColumn("qtd_votos_imdb_texto", F.trim(F.col("numVotes")))
)

df_metricas = (
    df_metricas
    .withColumn("popularidade_normalizada", normalizar_numero_decimal(F.col("popularidade_texto")))
    .withColumn("nota_media_tmdb_normalizada", normalizar_numero_decimal(F.col("nota_media_tmdb_texto")))
    .withColumn("qtd_votos_tmdb_normalizada", normalizar_numero_inteiro(F.col("qtd_votos_tmdb_texto")))
    .withColumn("nota_media_imdb_normalizada", normalizar_numero_decimal(F.col("nota_media_imdb_texto")))
    .withColumn("qtd_votos_imdb_normalizada", normalizar_numero_inteiro(F.col("qtd_votos_imdb_texto")))
    .withColumn("popularidade", F.expr("try_cast(popularidade_normalizada AS DOUBLE)"))
    .withColumn("nota_media_tmdb", F.expr("try_cast(nota_media_tmdb_normalizada AS DOUBLE)"))
    .withColumn("qtd_votos_tmdb", F.expr("try_cast(qtd_votos_tmdb_normalizada AS INT)"))
    .withColumn("nota_media_imdb", F.expr("try_cast(nota_media_imdb_normalizada AS DOUBLE)"))
    .withColumn("qtd_votos_imdb", F.expr("try_cast(qtd_votos_imdb_normalizada AS INT)"))
)

# valores fora do domínio sinalizam column shift ou sujeira e não devem contaminar as métricas
df_metricas = (
    df_metricas
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
)

# detecta o padrao observado de column shift sem impor um teto arbitrario de popularidade
df_referencia_metricas = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}").select(
    "id_filme", "titulo", "ano_lancamento"
)
df_metricas = df_metricas.join(df_referencia_metricas, on="id_filme", how="left")
popularidade_deslocada = (
    F.col("popularidade").isNotNull()
    & F.col("ano_lancamento").isNotNull()
    & (F.col("popularidade") == F.col("ano_lancamento").cast("double"))
)
df_popularidades_deslocadas = (
    df_metricas.where(popularidade_deslocada)
    .select("id_filme", "titulo", "ano_lancamento", "popularidade_texto", "popularidade")
    .dropDuplicates(["id_filme", "popularidade_texto", "ano_lancamento"])
)
df_metricas = (
    df_metricas
    .withColumn("popularidade", F.when(popularidade_deslocada, F.lit(None).cast("double")).otherwise(F.col("popularidade")))
    .drop("titulo", "ano_lancamento")
)

In [ ]:
colunas_metricas_silver = [
    "id_filme", "popularidade", "nota_media_tmdb",
    "qtd_votos_tmdb", "nota_media_imdb",
    "qtd_votos_imdb", "data_hora_ingestao"
]

# como a bronze é append-only, mantém a versão mais recente de cada filme
janela_metricas_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("data_hora_ingestao").desc())
df_metricas_silver = (
    df_metricas
    .where(F.col("id_filme").isNotNull())
    .withColumn("ordem_ingestao", F.row_number().over(janela_metricas_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .drop("ordem_ingestao", "id", "popularity", "vote_average",
          "vote_count", "averageRating", "numVotes",
          "popularidade_texto", "popularidade_normalizada",
          "nota_media_tmdb_texto", "nota_media_tmdb_normalizada",
          "qtd_votos_tmdb_texto", "qtd_votos_tmdb_normalizada",
          "nota_media_imdb_texto", "nota_media_imdb_normalizada",
          "qtd_votos_imdb_texto", "qtd_votos_imdb_normalizada")
    .select(*colunas_metricas_silver)
)

# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_metricas_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_METRICAS_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_METRICAS_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_METRICAS_DESTINO}").limit(10))

In [ ]:
# valida formatos, limites, valores nao negativos, column shift e unicidade por filme
df_metricas_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_METRICAS_DESTINO}")
validar_coluna_ingestao_silver(df_metricas_validacao, TABELA_METRICAS_DESTINO)
casos_teste_decimais = [
    ("2.994,357", "2994.357"),
    ("2,994.357", "2994.357"),
    ("2994,357", "2994.357"),
    ("2994.357", "2994.357"),
    ("-1,5", "-1.5"),
    ("texto", None),
]
df_teste_decimais = (
    spark.createDataFrame(casos_teste_decimais, ["valor_origem", "valor_esperado"])
    .withColumn("valor_normalizado", normalizar_numero_decimal(F.col("valor_origem")))
)
formatos_decimais_invalidos = df_teste_decimais.where(
    ~F.col("valor_normalizado").eqNullSafe(F.col("valor_esperado"))
).count()
casos_teste_inteiros = [
    ("1,234", "1234"),
    ("1.234", "1234"),
    ("1234.0", "1234"),
    ("1234", "1234"),
    ("texto", None),
]
df_teste_inteiros = (
    spark.createDataFrame(casos_teste_inteiros, ["valor_origem", "valor_esperado"])
    .withColumn("valor_normalizado", normalizar_numero_inteiro(F.col("valor_origem")))
)
formatos_inteiros_invalidos = df_teste_inteiros.where(
    ~F.col("valor_normalizado").eqNullSafe(F.col("valor_esperado"))
).count()
duplicados_metricas = (
    df_metricas_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
violacoes_notas = df_metricas_validacao.where(
    F.col("nota_media_tmdb").isNotNull() & ~F.col("nota_media_tmdb").between(0, 10)
).count() + df_metricas_validacao.where(
    F.col("nota_media_imdb").isNotNull() & ~F.col("nota_media_imdb").between(0, 10)
).count()
metricas_negativas = df_metricas_validacao.where(
    (F.col("popularidade").isNotNull() & (F.col("popularidade") < 0))
    | (F.col("qtd_votos_tmdb").isNotNull() & (F.col("qtd_votos_tmdb") < 0))
    | (F.col("qtd_votos_imdb").isNotNull() & (F.col("qtd_votos_imdb") < 0))
).count()
popularidades_iguais_ao_ano = (
    df_metricas_validacao
    .join(df_referencia_metricas.select("id_filme", "ano_lancamento"), on="id_filme", how="left")
    .where(
        F.col("popularidade").isNotNull()
        & F.col("ano_lancamento").isNotNull()
        & (F.col("popularidade") == F.col("ano_lancamento").cast("double"))
    )
    .count()
)

if formatos_decimais_invalidos + formatos_inteiros_invalidos != 0:
    raise AssertionError(
        f"Há {formatos_decimais_invalidos} formatos decimais e {formatos_inteiros_invalidos} formatos inteiros normalizados incorretamente."
    )
if duplicados_metricas != 0:
    raise AssertionError(f"Há {duplicados_metricas} ids de filme duplicados na Silver de métricas.")
if violacoes_notas != 0:
    raise AssertionError(f"Há {violacoes_notas} notas fora do intervalo permitido.")
if metricas_negativas != 0:
    raise AssertionError(f"Há {metricas_negativas} métricas negativas na Silver.")
if popularidades_iguais_ao_ano != 0:
    raise AssertionError(f"Há {popularidades_iguais_ao_ano} popularidades contaminadas pelo ano de lançamento.")

display(df_popularidades_deslocadas.orderBy("ano_lancamento", "titulo").limit(20))
display(df_metricas_validacao.select(
    "id_filme", "popularidade", "nota_media_tmdb",
    "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
).limit(10))
print(f"Registros Silver de métricas: {df_metricas_validacao.count()}")
print(f"Popularidades nulas ou inválidas: {df_metricas_validacao.where(F.col("popularidade").isNull()).count()}")
print(f"Notas nulas ou inválidas: {df_metricas_validacao.where(F.col("nota_media_tmdb").isNull() | F.col("nota_media_imdb").isNull()).count()}")
print(f"Duplicidades por id_filme: {duplicados_metricas}")
print(f"Popularidades removidas por coincidirem com o ano: {df_popularidades_deslocadas.count()}")
print(f"Formatos decimais inválidos nos testes: {formatos_decimais_invalidos}")
print(f"Formatos inteiros inválidos nos testes: {formatos_inteiros_invalidos}")
print(f"Métricas negativas restantes: {metricas_negativas}")

In [ ]:
SCHEMA_AVALIACOES_ORIGEM = "bronze"
TABELA_AVALIACOES_ORIGEM = "tb_movies_reviews"
TABELA_AVALIACOES_DESTINO = "tb_avaliacoes_usuarios"

df_avaliacoes_bronze = spark.table(f"{SCHEMA_AVALIACOES_ORIGEM}.{TABELA_AVALIACOES_ORIGEM}")
colunas_avaliacoes_esperadas = {"id", "nome", "nota", "comentario", "ingestion_datetime"}
colunas_avaliacoes_ausentes = colunas_avaliacoes_esperadas.difference(df_avaliacoes_bronze.columns)
if colunas_avaliacoes_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_avaliacoes_ausentes)}")

In [ ]:
# remove apenas duplicatas integrais da origem e conserva deterministicamente a ingestao mais recente
df_avaliacoes_origem = (
    df_avaliacoes_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
)
colunas_chave_avaliacao_origem = ["id_filme", "nome", "nota", "comentario"]
janela_avaliacao_mais_recente = Window.partitionBy(*colunas_chave_avaliacao_origem).orderBy(
    F.col("data_hora_ingestao").desc()
)
df_avaliacoes_sem_duplicatas = (
    df_avaliacoes_origem
    .withColumn("ordem_ingestao", F.row_number().over(janela_avaliacao_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .drop("ordem_ingestao")
)
id_avaliacao_valido = F.col("id_filme").isNotNull() & (F.col("id_filme") != "")
df_avaliacoes_sem_id = df_avaliacoes_sem_duplicatas.where(~id_avaliacao_valido)

# somente depois da deduplicacao aplica tipagem, limpeza e regras de negocio
df_avaliacoes = (
    df_avaliacoes_sem_duplicatas
    .where(id_avaliacao_valido)
    .withColumn("nome_usuario", F.trim(F.col("nome")))
    .withColumn("nota_usuario_normalizada", normalizar_numero_decimal(F.col("nota")))
    .withColumn("nota_usuario", F.expr("try_cast(nota_usuario_normalizada AS DOUBLE)"))
    .withColumn("comentario_usuario", F.trim(F.col("comentario")))
)

# notas fora do domínio de avaliação são inválidas e devem ficar como NULL
df_avaliacoes = df_avaliacoes.withColumn(
    "nota_usuario",
    F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario"))
)

# comentários nulos ou somente com espaços recebem um texto padrão
df_avaliacoes = df_avaliacoes.withColumn(
    "comentario_usuario",
    F.when(F.col("comentario_usuario").isNull() | (F.col("comentario_usuario") == ""), F.lit("Sem comentário"))
     .otherwise(F.col("comentario_usuario"))
)

colunas_avaliacoes_silver = [
    "id_filme", "nome_usuario", "nota_usuario", "comentario_usuario", "data_hora_ingestao"
]
df_avaliacoes_silver = (
    df_avaliacoes
    .select(*colunas_avaliacoes_silver)
)

In [ ]:
# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_avaliacoes_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_AVALIACOES_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_AVALIACOES_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_AVALIACOES_DESTINO}").limit(10))

In [ ]:
# valida notas, comentarios, ids e deduplicacao integral realizada antes da normalizacao
df_avaliacoes_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_AVALIACOES_DESTINO}")
validar_coluna_ingestao_silver(df_avaliacoes_validacao, TABELA_AVALIACOES_DESTINO)
duplicados_avaliacoes_origem = (
    df_avaliacoes_sem_duplicatas
    .groupBy(*colunas_chave_avaliacao_origem).count()
    .where(F.col("count") > 1)
    .count()
)
quantidade_duplicatas_removidas = df_avaliacoes_origem.count() - df_avaliacoes_sem_duplicatas.count()
quantidade_avaliacoes_sem_id = df_avaliacoes_sem_id.count()
quantidade_avaliacoes_esperada = df_avaliacoes_sem_duplicatas.where(id_avaliacao_valido).count()
quantidade_avaliacoes_gravada = df_avaliacoes_validacao.count()
ids_avaliacoes_invalidos = df_avaliacoes_validacao.where(
    F.col("id_filme").isNull() | (F.trim(F.col("id_filme")) == "")
).count()
violacoes_notas_avaliacoes = df_avaliacoes_validacao.where(
    F.col("nota_usuario").isNotNull() & ~F.col("nota_usuario").between(0, 10)
).count()
comentarios_vazios = df_avaliacoes_validacao.where(
    F.col("comentario_usuario").isNull() | (F.trim(F.col("comentario_usuario")) == "")
).count()

if duplicados_avaliacoes_origem != 0:
    raise AssertionError(f"Há {duplicados_avaliacoes_origem} avaliações integralmente duplicadas após o tratamento.")
if quantidade_avaliacoes_gravada != quantidade_avaliacoes_esperada:
    raise AssertionError(
        f"A Silver possui {quantidade_avaliacoes_gravada} avaliações, mas eram esperadas {quantidade_avaliacoes_esperada}."
    )
if ids_avaliacoes_invalidos != 0:
    raise AssertionError(f"Há {ids_avaliacoes_invalidos} avaliações sem id_filme válido na Silver.")
if violacoes_notas_avaliacoes != 0:
    raise AssertionError(f"Há {violacoes_notas_avaliacoes} notas fora do intervalo permitido.")
if comentarios_vazios != 0:
    raise AssertionError(f"Há {comentarios_vazios} comentários vazios na Silver.")

display(df_avaliacoes_sem_id.select("id", "nome", "nota", "comentario").limit(20))
display(df_avaliacoes_validacao.select(*colunas_avaliacoes_silver).limit(10))
print(f"Registros Silver de avaliações: {quantidade_avaliacoes_gravada}")
print(f"Notas nulas ou inválidas: {df_avaliacoes_validacao.where(F.col('nota_usuario').isNull()).count()}")
print(f"Comentários padronizados: {df_avaliacoes_validacao.where(F.col('comentario_usuario') == 'Sem comentário').count()}")
print(f"Duplicatas integrais removidas: {quantidade_duplicatas_removidas}")
print(f"Avaliações descartadas por id_filme ausente: {quantidade_avaliacoes_sem_id}")
print(f"Duplicatas integrais restantes: {duplicados_avaliacoes_origem}")

In [ ]:
SCHEMA_GENEROS_ORIGEM = "bronze"
TABELA_GENEROS_ORIGEM = "tb_credits_and_tags"
TABELA_GENEROS_DESTINO = "tb_generos"

df_generos_bronze = spark.table(f"{SCHEMA_GENEROS_ORIGEM}.{TABELA_GENEROS_ORIGEM}")
colunas_generos_esperadas = {"id", "genres", "ingestion_datetime"}
colunas_generos_ausentes = colunas_generos_esperadas.difference(df_generos_bronze.columns)
if colunas_generos_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_generos_ausentes)}")

In [ ]:
# normaliza separadores para que listas com vírgula, ponto e vírgula ou barra vertical tenham o mesmo tratamento
df_generos = (
    df_generos_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("generos_texto", F.trim(F.coalesce(F.col("genres"), F.lit(""))))
    .withColumn("generos_normalizados", F.regexp_replace(F.col("generos_texto"), r"\s*[;|]\s*", ","))
)

# explode transforma a lista em relações filme-gênero sem alterar a tabela bronze
df_generos = (
    df_generos
    .withColumn("genero_bruto", F.explode(F.split(F.col("generos_normalizados"), ",")))
    .withColumn("nome_genero", F.trim(F.regexp_replace(F.col("genero_bruto"), r"[\[\]{}\"]", "")))
)

# restringe os valores ao domínio oficial de gêneros do TMDB para remover resíduos de Column Shift
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western",
]
mapa_generos_validos = {genero.lower(): genero for genero in generos_validos}
mapa_generos_expr = F.create_map(
    *[item for chave, valor in mapa_generos_validos.items() for item in (F.lit(chave), F.lit(valor))]
)
marcadores_genero_ausente = ["", "n/a", "na", "null", "unknown", "não informado", "[]"]
df_generos = (
    df_generos
    .withColumn("nome_genero", F.trim(F.col("nome_genero")))
    .where(F.col("id_filme").isNotNull())
    .where(F.length(F.col("nome_genero")) > 0)
    .where(~F.lower(F.col("nome_genero")).isin(marcadores_genero_ausente))
    .withColumn("nome_genero", mapa_generos_expr[F.lower(F.col("nome_genero"))])
    .where(F.col("nome_genero").isNotNull())
)

In [ ]:
# como a bronze é append-only, conserva a versão mais recente de cada relação filme-gênero
janela_genero_mais_recente = Window.partitionBy("id_filme", "nome_genero").orderBy(F.col("data_hora_ingestao").desc())
colunas_generos_silver = ["id_filme", "nome_genero", "data_hora_ingestao"]
df_generos_silver = (
    df_generos
    .withColumn("ordem_ingestao", F.row_number().over(janela_genero_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .select(*colunas_generos_silver)
)

# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_generos_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_GENEROS_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_GENEROS_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_GENEROS_DESTINO}").limit(10))

In [ ]:
# valida que cada linha contém um gênero limpo e que a relação filme-gênero não se repete
df_generos_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_GENEROS_DESTINO}")
validar_coluna_ingestao_silver(df_generos_validacao, TABELA_GENEROS_DESTINO)
duplicados_generos = (
    df_generos_validacao.groupBy("id_filme", "nome_genero").count().where(F.col("count") > 1).count()
)
generos_vazios = df_generos_validacao.where(
    F.col("nome_genero").isNull() | (F.trim(F.col("nome_genero")) == "")
).count()
generos_fora_dominio = df_generos_validacao.where(
    ~F.col("nome_genero").isin(generos_validos)
).count()

if duplicados_generos != 0:
    raise AssertionError(f"Há {duplicados_generos} relações filme-gênero duplicadas na Silver.")
if generos_vazios != 0:
    raise AssertionError(f"Há {generos_vazios} gêneros vazios na Silver.")
if generos_fora_dominio != 0:
    raise AssertionError(f"Há {generos_fora_dominio} valores fora do domínio de gêneros na Silver.")

display(df_generos_validacao.groupBy("nome_genero").count().orderBy(F.col("count").desc()))
print(f"Registros Silver de gêneros: {df_generos_validacao.count()}")
print(f"Filmes com gênero: {df_generos_validacao.select('id_filme').distinct().count()}")
print(f"Duplicidades filme-gênero: {duplicados_generos}")
print(f"Valores fora do domínio de gêneros: {generos_fora_dominio}")

In [ ]:
SCHEMA_PESSOAS_ORIGEM = "bronze"
TABELA_PESSOAS_ORIGEM = "tb_credits_and_tags"
TABELA_PESSOAS_DESTINO = "tb_pessoas_empresas"

df_pessoas_bronze = spark.table(f"{SCHEMA_PESSOAS_ORIGEM}.{TABELA_PESSOAS_ORIGEM}")
colunas_pessoas_esperadas = {"id", "cast", "directors", "writers", "production_companies", "ingestion_datetime"}
colunas_pessoas_ausentes = colunas_pessoas_esperadas.difference(df_pessoas_bronze.columns)
if colunas_pessoas_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_pessoas_ausentes)}")

In [ ]:
# normaliza listas, marcadores de ausência e identificadores sem alterar a bronze
df_pessoas = (
    df_pessoas_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("atores_texto", F.trim(F.coalesce(F.col("cast"), F.lit(""))))
    .withColumn("diretores_texto", F.trim(F.coalesce(F.col("directors"), F.lit(""))))
    .withColumn("roteiristas_texto", F.trim(F.coalesce(F.col("writers"), F.lit(""))))
    .withColumn("produtoras_texto", F.trim(F.coalesce(F.col("production_companies"), F.lit(""))))
)

# unpivot consolida as quatro origens em uma dimensão com o tipo da entidade
df_pessoas = df_pessoas.select(
    "id_filme", "data_hora_ingestao",
    F.expr("stack(4, 'Ator', atores_texto, 'Diretor', diretores_texto, 'Roteirista', roteiristas_texto, 'Produtora', produtoras_texto) AS (tipo_entidade, entidades_texto)")
)

# preserva nomes mistos e normaliza somente textos inteiramente em caixa alta ou baixa
def padronizar_nome_entidade(coluna):
    texto = F.trim(F.regexp_replace(coluna, r"\s+", " "))
    tokens = F.split(texto, " ")
    tokens_minusculos = F.transform(
        tokens,
        lambda token: F.when(F.instr(token, ".") > 0, F.upper(token)).otherwise(F.initcap(token)),
    )
    tokens_maiusculos = F.transform(
        tokens,
        lambda token: F.when(
            (F.length(token) <= 3) | (F.instr(token, ".") > 0), token
        ).otherwise(F.initcap(F.lower(token))),
    )
    return (
        F.when(texto == F.lower(texto), F.concat_ws(" ", tokens_minusculos))
         .when(texto == F.upper(texto), F.concat_ws(" ", tokens_maiusculos))
         .otherwise(texto)
    )

marcadores_entidade_ausente = ["", "n/a", "na", "null", "unknown", "não informado", "[]"]

def classificar_residuo_entidade(coluna):
    texto = F.lower(F.trim(coluna))
    return (
        F.when(texto.isNull() | texto.isin(marcadores_entidade_ausente), F.lit("marcador de ausência"))
         .when(texto.rlike(r"^[+-]?[0-9]+([.,][0-9]+)*$"), F.lit("valor numérico deslocado"))
         .when(texto.rlike(r"^[0-9]{4}-[0-9]{1,2}-[0-9]{1,2}([ t].*)?$"), F.lit("data deslocada"))
         .when(texto.rlike(r"^[0-9]{1,2}[/-][0-9]{1,2}[/-][0-9]{2,4}([ t].*)?$"), F.lit("data deslocada"))
         .when(~texto.rlike(r".*\p{L}.*"), F.lit("texto sem letras"))
         .when(texto.rlike(r"^(https?://|www\.).*") | texto.rlike(r"^.*@.*\..*$"), F.lit("link ou contato deslocado"))
         .when(texto.rlike(r"^(released|post production|in production|planned|rumored|canceled|cancelled|lançado|pós-produção|em produção|planejado|rumores|cancelado|true|false)$"), F.lit("atributo contextual deslocado"))
         .when(F.length(texto) > 150, F.lit("texto descritivo extenso"))
         .otherwise(F.lit(None).cast("string"))
    )

# separa as listas antes do explode e registra a justificativa de cada descarte
df_pessoas = (
    df_pessoas
    .withColumn("entidade_bruta", F.explode(F.split(F.col("entidades_texto"), r"\s*[,;|]\s*")))
    .withColumn("nome_entidade_base", F.trim(F.regexp_replace(F.col("entidade_bruta"), r"[\[\]{}\"]", "")))
    .withColumn("nome_entidade_base", F.regexp_replace(F.col("nome_entidade_base"), r"\s+", " "))
    .withColumn("motivo_rejeicao",
                F.when(F.col("id_filme").isNull(), F.lit("id_filme ausente"))
                 .otherwise(classificar_residuo_entidade(F.col("nome_entidade_base"))))
    .withColumn("nome_entidade", padronizar_nome_entidade(F.col("nome_entidade_base")))
)
df_entidades_rejeitadas = df_pessoas.where(F.col("motivo_rejeicao").isNotNull()).select(
    "id_filme", "tipo_entidade", "entidade_bruta", "motivo_rejeicao"
)
df_pessoas = df_pessoas.where(F.col("motivo_rejeicao").isNull())

# mantém a versão mais recente e remove entidades repetidas no mesmo filme e tipo
janela_pessoa_mais_recente = Window.partitionBy("id_filme", "nome_entidade", "tipo_entidade").orderBy(F.col("data_hora_ingestao").desc())
colunas_pessoas_silver = ["id_filme", "nome_entidade", "tipo_entidade", "data_hora_ingestao"]
df_pessoas_silver = (
    df_pessoas
    .withColumn("ordem_ingestao", F.row_number().over(janela_pessoa_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .select(*colunas_pessoas_silver)
)

In [ ]:
# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_pessoas_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_PESSOAS_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_PESSOAS_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_PESSOAS_DESTINO}").limit(10))

In [ ]:
# valida tipos permitidos, nomes, capitalizacao, residuos e ausencia de duplicatas
df_pessoas_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_PESSOAS_DESTINO}")
validar_coluna_ingestao_silver(df_pessoas_validacao, TABELA_PESSOAS_DESTINO)
casos_teste_nomes = [
    ("BBC", "BBC"),
    ("UNIVERSAL PICTURES", "Universal Pictures"),
    ("J.J. ABRAMS", "J.J. Abrams"),
    ("kevin hart", "Kevin Hart"),
    ("Kevin McCarthy", "Kevin McCarthy"),
]
df_teste_nomes = (
    spark.createDataFrame(casos_teste_nomes, ["nome_origem", "nome_esperado"])
    .withColumn("nome_padronizado", padronizar_nome_entidade(F.col("nome_origem")))
)
formatos_nomes_invalidos = df_teste_nomes.where(
    ~F.col("nome_padronizado").eqNullSafe(F.col("nome_esperado"))
).count()
duplicados_pessoas = (
    df_pessoas_validacao.groupBy("id_filme", "nome_entidade", "tipo_entidade").count().where(F.col("count") > 1).count()
)
tipos_invalidos = df_pessoas_validacao.where(
    ~F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista", "Produtora")
).count()
entidades_vazias = df_pessoas_validacao.where(
    F.col("nome_entidade").isNull() | (F.trim(F.col("nome_entidade")) == "")
).count()
entidades_residuais = (
    df_pessoas_validacao
    .withColumn("motivo_rejeicao", classificar_residuo_entidade(F.col("nome_entidade")))
    .where(F.col("motivo_rejeicao").isNotNull())
    .count()
)
nomes_com_espacos_irregulares = df_pessoas_validacao.where(
    F.col("nome_entidade") != F.trim(F.regexp_replace(F.col("nome_entidade"), r"\s+", " "))
).count()

if formatos_nomes_invalidos != 0:
    raise AssertionError(f"Há {formatos_nomes_invalidos} nomes padronizados incorretamente nos testes.")
if duplicados_pessoas != 0:
    raise AssertionError(f"Há {duplicados_pessoas} entidades duplicadas na Silver.")
if tipos_invalidos != 0:
    raise AssertionError(f"Há {tipos_invalidos} tipos de entidade inválidos na Silver.")
if entidades_vazias != 0:
    raise AssertionError(f"Há {entidades_vazias} entidades vazias na Silver.")
if entidades_residuais != 0:
    raise AssertionError(f"Há {entidades_residuais} resíduos de entidades na Silver.")
if nomes_com_espacos_irregulares != 0:
    raise AssertionError(f"Há {nomes_com_espacos_irregulares} nomes com espaços irregulares na Silver.")

display(df_entidades_rejeitadas.groupBy("motivo_rejeicao").count().orderBy(F.col("count").desc()))
display(df_entidades_rejeitadas.orderBy("motivo_rejeicao", "tipo_entidade").limit(20))
display(df_pessoas_validacao.groupBy("tipo_entidade").count().orderBy(F.col("count").desc()))
print(f"Registros Silver de pessoas e empresas: {df_pessoas_validacao.count()}")
print(f"Filmes relacionados: {df_pessoas_validacao.select('id_filme').distinct().count()}")
print(f"Duplicidades filme-entidade-tipo: {duplicados_pessoas}")
print(f"Entidades rejeitadas durante a limpeza: {df_entidades_rejeitadas.count()}")
print(f"Resíduos restantes na Silver: {entidades_residuais}")
print(f"Falhas nos testes de capitalização: {formatos_nomes_invalidos}")

In [ ]:
SCHEMA_COTACAO_ORIGEM = "bronze"
TABELA_COTACAO_ORIGEM = "tb_cotacao_dolar"
TABELA_COTACAO_DESTINO = "tb_cotacao_dolar"

df_cotacao_bronze = spark.table(f"{SCHEMA_COTACAO_ORIGEM}.{TABELA_COTACAO_ORIGEM}")
colunas_cotacao_esperadas = {"dataHoraCotacao", "cotacaoCompra", "ingestion_datetime"}
colunas_cotacao_ausentes = colunas_cotacao_esperadas.difference(df_cotacao_bronze.columns)
if colunas_cotacao_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_cotacao_ausentes)}")

# usa os limites dos widgets quando a Bronze já possui os metadados adicionados pela Landing atual
colunas_periodo = {"data_inicio_periodo", "data_fim_periodo"}
periodo_solicitado = None
if colunas_periodo.issubset(df_cotacao_bronze.columns):
    periodo_solicitado = (
        df_cotacao_bronze
        .where(F.col("data_inicio_periodo").isNotNull() & F.col("data_fim_periodo").isNotNull())
        .orderBy(F.col("ingestion_datetime").desc())
        .select(
            F.to_date("data_inicio_periodo").alias("data_inicio"),
            F.to_date("data_fim_periodo").alias("data_fim"),
        )
        .first()
    )

if periodo_solicitado is None:
    print("Aviso: limites dos widgets ausentes na Bronze; será usado o intervalo observado nas cotações.")

In [ ]:
# converte o horário da API e mantém somente cotações válidas
df_cotacao_diaria = (
    df_cotacao_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("data_hora_cotacao", F.to_timestamp("dataHoraCotacao"))
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
    .withColumn("cotacao_dolar_brl", F.col("cotacaoCompra").cast("DECIMAL(12,6)"))
    .where(F.col("data_cotacao").isNotNull())
    .where(F.col("cotacao_dolar_brl") > 0)
)

# conserva a última cotação disponível em cada dia
janela_cotacao_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("data_hora_cotacao").desc(),
    F.col("data_hora_ingestao").desc(),
)
df_cotacao_diaria = (
    df_cotacao_diaria
    .withColumn("ordem_cotacao", F.row_number().over(janela_cotacao_dia))
    .where(F.col("ordem_cotacao") == 1)
    .select("data_cotacao", "cotacao_dolar_brl", "data_hora_ingestao")
)

# tabelas Bronze antigas não possuem os limites dos widgets; nesse caso, usa o intervalo observado
limites_observados = df_cotacao_diaria.agg(
    F.min("data_cotacao").alias("data_minima"),
    F.max("data_cotacao").alias("data_maxima"),
).first()
if limites_observados["data_minima"] is None:
    raise ValueError("A origem Bronze não possui cotações válidas.")

if periodo_solicitado is None:
    data_inicio_periodo = limites_observados["data_minima"]
    data_fim_periodo = limites_observados["data_maxima"]
else:
    data_inicio_periodo = periodo_solicitado["data_inicio"]
    data_fim_periodo = periodo_solicitado["data_fim"]

if data_inicio_periodo > data_fim_periodo:
    raise ValueError("O período de cotação possui limites inválidos.")

# localiza a última cotação conhecida até o início solicitado para semear o forward fill
data_semente = (
    df_cotacao_diaria
    .where(F.col("data_cotacao") <= F.lit(data_inicio_periodo))
    .agg(F.max("data_cotacao").alias("data_semente"))
    .first()["data_semente"]
)
if data_semente is None:
    raise ValueError("Não há cotação-semente anterior ou igual ao início do período solicitado.")

# inclui a data-semente no calendário temporário e recorta o resultado ao intervalo dos widgets
df_calendario_cotacao = spark.range(1).select(
    F.explode(
        F.sequence(F.lit(data_semente), F.lit(data_fim_periodo))
    ).alias("data_cotacao")
)

# preenche fins de semana e feriados com a última cotação útil conhecida
janela_forward_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_cotacao_silver = (
    df_calendario_cotacao
    .join(df_cotacao_diaria, on="data_cotacao", how="left")
    .withColumn("data_origem_cotacao",
                F.when(F.col("cotacao_dolar_brl").isNotNull(), F.col("data_cotacao")))
    .withColumn("cotacao_dolar_brl", F.last("cotacao_dolar_brl", ignorenulls=True).over(janela_forward_fill))
    .withColumn("data_hora_ingestao", F.last("data_hora_ingestao", ignorenulls=True).over(janela_forward_fill))
    .withColumn("data_origem_cotacao", F.last("data_origem_cotacao", ignorenulls=True).over(janela_forward_fill))
    .withColumn("cotacao_preenchida", F.col("data_origem_cotacao") < F.col("data_cotacao"))
    .where(F.col("data_cotacao").between(F.lit(data_inicio_periodo), F.lit(data_fim_periodo)))
    .select("data_cotacao", "data_origem_cotacao", "cotacao_preenchida",
            "cotacao_dolar_brl", "data_hora_ingestao")
    .orderBy("data_cotacao")
)

In [ ]:
# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_cotacao_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_COTACAO_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_COTACAO_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_COTACAO_DESTINO}").orderBy("data_cotacao").limit(10))

In [ ]:
# valida limites solicitados, continuidade, cotacao-semente, forward fill e ausencia de duplicatas
df_cotacao_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_COTACAO_DESTINO}")
validar_coluna_ingestao_silver(df_cotacao_validacao, TABELA_COTACAO_DESTINO)
duplicados_cotacao = df_cotacao_validacao.groupBy("data_cotacao").count().where(F.col("count") > 1).count()
datas_cotacao = df_cotacao_validacao.agg(
    F.min("data_cotacao").alias("data_minima"),
    F.max("data_cotacao").alias("data_maxima"),
    F.count("data_cotacao").alias("quantidade_datas"),
).first()
if datas_cotacao["data_minima"] is None:
    raise AssertionError("A tabela Silver de cotação foi gravada sem registros.")
quantidade_datas_esperada = (data_fim_periodo - data_inicio_periodo).days + 1
cotacoes_vazias = df_cotacao_validacao.where(F.col("cotacao_dolar_brl").isNull()).count()
primeira_cotacao = df_cotacao_validacao.orderBy("data_cotacao").first()
origens_invalidas = df_cotacao_validacao.where(
    F.col("data_origem_cotacao").isNull()
    | (F.col("data_origem_cotacao") > F.col("data_cotacao"))
).count()
marcadores_preenchimento_incorretos = df_cotacao_validacao.where(
    F.col("cotacao_preenchida") != (F.col("data_origem_cotacao") < F.col("data_cotacao"))
).count()

if duplicados_cotacao != 0:
    raise AssertionError(f"Há {duplicados_cotacao} datas de cotação duplicadas na Silver.")
if datas_cotacao["data_minima"] != data_inicio_periodo or datas_cotacao["data_maxima"] != data_fim_periodo:
    raise AssertionError("A Silver não cobre exatamente o período informado nos widgets.")
if datas_cotacao["quantidade_datas"] != quantidade_datas_esperada:
    raise AssertionError("O calendário de cotações possui datas faltantes.")
if cotacoes_vazias != 0:
    raise AssertionError(f"Há {cotacoes_vazias} datas sem cotação preenchida na Silver.")
if primeira_cotacao["data_origem_cotacao"] != data_semente:
    raise AssertionError("A primeira data do período não usa a cotação-semente esperada.")
if origens_invalidas != 0:
    raise AssertionError(f"Há {origens_invalidas} datas preenchidas sem uma cotação anterior válida.")
if marcadores_preenchimento_incorretos != 0:
    raise AssertionError(f"Há {marcadores_preenchimento_incorretos} marcadores de forward fill incorretos.")

display(df_cotacao_validacao.where(F.col("cotacao_preenchida")).orderBy("data_cotacao"))
display(df_cotacao_validacao.orderBy("data_cotacao").limit(10))
print(f"Período solicitado: {data_inicio_periodo} a {data_fim_periodo}")
print(f"Data da cotação-semente: {data_semente}")
print(f"Registros Silver de cotação: {datas_cotacao['quantidade_datas']}")
print(f"Datas sem cotação: {cotacoes_vazias}")
print(f"Duplicidades por data: {duplicados_cotacao}")
print(f"Datas preenchidas por forward fill: {df_cotacao_validacao.where(F.col('cotacao_preenchida')).count()}")
print(f"Origens de cotação inválidas: {origens_invalidas}")